# Лабораторная работа №1

**Дисциплина:** Инструменты хранения и анализа больших данных  
**Студент:** Войт Иван Иванович  
**Группа:** Бд-251м  
**Вариант:** 10. Ценообразование

## Постановка задачи

Необходимо исследовать, как изменение цены влияет на объем спроса. Для этого используются данные о товарах, ценах и количестве продаж. Анализ выполняется в `Google Colab` с использованием `PySpark`.

**Источник данных:** Kaggle dataset `suddharshan/retail-price-optimization`.


## Шаг 1. Установка библиотек

Устанавливаем библиотеки для работы с `PySpark`, табличными данными, графиками и Kaggle-датасетом.


In [ ]:
!pip install -q pyspark pandas matplotlib numpy kagglehub

## Шаг 2. Загрузка датасета из Kaggle

Скачиваем датасет и автоматически находим CSV-файл, который будем анализировать.


In [ ]:
import kagglehub
from pathlib import Path

# Скачиваем датасет из Kaggle в среду Google Colab
dataset_path = kagglehub.dataset_download("suddharshan/retail-price-optimization")

# Ищем первый CSV-файл внутри скачанной папки
csv_path = str(next(Path(dataset_path).rglob("*.csv")))

print("Папка с датасетом:", dataset_path)
print("Путь к CSV:", csv_path)

## Шаг 3. Инициализация Spark и первичная загрузка данных

Создаем `SparkSession`, считываем CSV в `DataFrame`, смотрим структуру таблицы и первые строки.


In [ ]:
from pyspark.sql import SparkSession

# Создаем Spark-сессию для локального выполнения в среде Colab
spark = SparkSession.builder \
    .appName("Lab1_PricingElasticity_Colab") \
    .master("local[*]") \
    .getOrCreate()

# Загружаем CSV-файл в Spark DataFrame
raw_df = spark.read.option("header", "true").option("inferSchema", "true").csv(csv_path)

# Выводим схему и первые строки для знакомства с данными
raw_df.printSchema()
raw_df.show(5, truncate=False)

## Шаг 4. Очистка и предобработка данных

Оставляем только нужные для исследования поля, приводим типы, удаляем пустые значения и некорректные записи.


In [ ]:
from pyspark.sql.functions import avg, col, corr, count, round as spark_round, sum as spark_sum, when

# Выбираем только нужные столбцы для анализа ценообразования
df = (
    raw_df.select(
        col("product_id"),
        col("product_category_name").alias("category"),
        col("month_year"),
        col("qty").cast("double").alias("units_sold"),
        col("unit_price").cast("double").alias("price"),
        col("total_price").cast("double").alias("total_price"),
        col("freight_price").cast("double").alias("freight_price"),
        col("customers").cast("double").alias("customers"),
        col("month").cast("int").alias("month"),
        col("year").cast("int").alias("year")
    )
    # Удаляем строки, где отсутствуют ключевые поля
    .dropna(subset=["product_id", "category", "units_sold", "price"])
    # Оставляем только корректные наблюдения
    .filter((col("price") > 0) & (col("units_sold") > 0))
    # Удаляем возможные дубликаты
    .dropDuplicates()
    # Рассчитываем оценочную выручку
    .withColumn("estimated_revenue", spark_round(col("price") * col("units_sold"), 2))
)

df.printSchema()
df.show(5, truncate=False)

## Шаг 5. Проверка пропусков

Показываем количество пропусков по ключевым полям после очистки. Это пригодится для отчета.


In [ ]:
# Подсчитываем количество NULL по основным полям
null_check = raw_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in [
        "product_id", "product_category_name", "qty", "unit_price", "month_year"
    ]
])

null_check.show()

## Шаг 6. Корреляционный анализ

Считаем общую корреляцию между ценой и объемом продаж, а затем сравниваем категории товаров.


In [ ]:
# Общая корреляция между ценой и количеством продаж
overall_corr = df.stat.corr("price", "units_sold")
print(f"Общая корреляция между ценой и объемом продаж: {overall_corr:.4f}")

# Корреляция и агрегаты по категориям товаров
elasticity_df = (
    df.groupBy("category")
      .agg(
          count("*").alias("observations"),
          spark_round(avg("price"), 2).alias("avg_price"),
          spark_round(avg("units_sold"), 2).alias("avg_units_sold"),
          spark_round(corr("price", "units_sold"), 4).alias("price_qty_corr"),
          spark_round(spark_sum("estimated_revenue"), 2).alias("estimated_revenue")
      )
      .orderBy("price_qty_corr")
)

elasticity_df.show(truncate=False)

## Шаг 7. Spark SQL-анализ

Регистрируем таблицу и выполняем SQL-запрос для получения бизнес-показателей по категориям.


In [ ]:
# Создаем временное представление для SQL-запросов
df.createOrReplaceTempView("pricing_data")

# Выполняем SQL-анализ по категориям
sql_result = spark.sql("""
SELECT
    category,
    COUNT(*) AS observations,
    ROUND(AVG(price), 2) AS avg_price,
    ROUND(AVG(units_sold), 2) AS avg_units_sold,
    ROUND(CORR(price, units_sold), 4) AS price_qty_corr,
    ROUND(SUM(estimated_revenue), 2) AS estimated_revenue
FROM pricing_data
GROUP BY category
ORDER BY price_qty_corr ASC
""")

sql_result.show(truncate=False)

## Шаг 8. Визуализация

Строим график рассеяния `Цена vs Объем продаж` и добавляем линию регрессии.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Переводим часть данных в pandas для построения графика
plot_df = df.select("price", "units_sold").toPandas()

# Рассчитываем линейную регрессию
coeffs = np.polyfit(plot_df["price"], plot_df["units_sold"], deg=1)
x_line = np.linspace(plot_df["price"].min(), plot_df["price"].max(), 200)
y_line = coeffs[0] * x_line + coeffs[1]

# Строим scatter plot и линию регрессии
plt.figure(figsize=(10, 6))
plt.scatter(plot_df["price"], plot_df["units_sold"], alpha=0.65, s=42, color="#2a9d8f")
plt.plot(x_line, y_line, color="#d62828", linewidth=2.2, label="Линия регрессии")
plt.title(f"Цена vs Объем продаж (corr = {plot_df['price'].corr(plot_df['units_sold']):.4f})")
plt.xlabel("Цена товара")
plt.ylabel("Объем продаж")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

## Шаг 9. Интерпретация результатов

- Если коэффициент корреляции отрицательный, то с ростом цены объем продаж уменьшается.
- Чем ближе значение к `-1`, тем сильнее выражена ценовая эластичность.
- Нисходящая линия регрессии подтверждает обратную зависимость между ценой и спросом.
- Анализ по категориям позволяет понять, какие группы товаров наиболее чувствительны к изменению цены.
